In [1]:
import re
import warnings
import pyreadr
import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
import statsmodels.tools.sm_exceptions as sm_exceptions
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
# Setting output directory
output = "C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/09assoc-08282025/"

In [3]:
# loading phenotype data
phenoPath = "C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/00databases/00phenotype/01uthealth/00MetaData_UTHealth_RNAseq-07292025.csv"
phenoData = pd.read_csv(phenoPath)
phenoData.rename(columns={phenoData.columns[0]: "SampleID"}, inplace=True)
phenoData = phenoData[['SampleID', 'SAB', 'UTID', 'RIN_novogene', 'BA9_RNAseq_Batch']]
phenoData
# Load phenotypes
pheno = pd.read_csv('C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/00databases/00phenotype/01uthealth/01MetaData_UTHealth_Methylation-08052024.csv')
# Merge on 'SampleID'
phenoData = pd.merge(phenoData, pheno, on='SAB')
# Load data
uthh = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/02trans_age-07272025/00uthhealth_rnaagecalc-07272025.csv")
# Rename the first column to SampleID
uthh.rename(columns={uthh.columns[0]: "SampleID"}, inplace=True)
uthh.rename(
    columns={col: f"{col}_clock" for col in uthh.columns if col != "SampleID"},
    inplace=True
)
# Load data
dpclo = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/00deepclock-08032025.csv")
# Rename and keep only SampleID and KPANN_brain columns
dpclo = dpclo.rename(columns={'ID': 'SampleID', 'Predicted': 'KPANN_brain_clock'})[['SampleID', 'KPANN_brain_clock']]
# Merge with both filtered datasets
merged_df_com = pd.merge(uthh, dpclo, on='SampleID')
# Load cell types and SVAs
cells = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/05celltype-08042025/01uthealth_cellprop_svas-08042025.csv")
cells = cells.rename(columns={'name': 'SampleID'})
# Merge with both filtered datasets
merged_df_com = pd.merge(merged_df_com, cells, on='SampleID')
# Load KPANN clock trained in VABB for UTHealth
adicloc = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/01deepclock_UTHealth_only-08052025.csv")
adicloc = adicloc.rename(columns={'name': 'SampleID'})
# Merge with both filtered datasets
merged_df_com = pd.merge(merged_df_com, adicloc, on='SampleID')
# Merge on SampleID
data_assoc = pd.merge(phenoData, merged_df_com, on="SampleID", how="inner")
data_assoc.head()

,SampleID,SAB,UTID,RIN_novogene,BA9_RNAseq_Batch,Sentrix_ID,Year,Ethnicity,Gender,Age,...,W_2,W_3,W_4,W_5,W_6,W_7,W_8,W_9,W_10,KPANNtrainedVABB_brain_clock
0,A67900,67900,UTHBC0001,6.9,B2018,202787550060_R01C01,2018,White,Male,17,...,-0.011269,-0.032170,0.000214,-0.033074,0.001498,-0.049769,0.002038,0.004475,-0.029818,42.436069
1,A67902,67902,UTHBC0003,7.2,B2018,202787550060_R03C01,2018,Hispanic,Male,31,...,-0.002684,-0.009380,0.026539,-0.102646,0.045524,-0.048069,-0.167566,-0.106294,0.069254,48.787434
2,A67905,67905,UTHBC0006,8.4,B2018,202787550060_R04C01,2018,White,Male,43,...,-0.003179,-0.001653,-0.045735,-0.043083,-0.008536,0.033994,-0.014402,0.002389,0.036657,40.489330
3,A67906,67906,UTHBC0007,7.5,B2018,202787550060_R05C01,2018,White,Male,44,...,-0.007012,0.026808,0.028630,-0.036488,0.000311,-0.034601,0.024915,-0.043130,0.009967,44.178295
4,A67907,67907,UTHBC0008,7.6,B2018,202787550060_R06C01,2018,White,Male,53,...,0.003290,-0.034876,0.028161,0.024909,0.056005,0.014285,-0.107466,0.156924,0.006903,42.630310


In [13]:
# Load dataset
uth = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/06tranningclock-08152025/00RetrainedPredictions_uthealth-08182025.csv")
# Rename the first column to SampleID
uth = uth.rename(columns={uth.columns[0]: "SampleID"})
uth = uth.rename(columns={col: f"{col}_brain" for col in uth.columns if col != "SampleID"})
# Merge on SampleID
data_assoc = pd.merge(data_assoc, uth, on="SampleID", how="inner")
data_assoc.shape

(80, 259)

In [5]:
# Standarizing all covariates
# Select only numeric columns
numeric_cols = data_assoc.select_dtypes(include='number').columns
# Scale
scaler = StandardScaler()
data_assoc[numeric_cols] = scaler.fit_transform(data_assoc[numeric_cols])

In [6]:
# Copy your data
binary_df = data_assoc.copy()

# Define clocks and phenotype columns
brain_clocks = [col for col in binary_df.columns 
                if "brain" in col.lower() and pd.api.types.is_numeric_dtype(binary_df[col])]
# Convert Gender (and any other categorical covariates) to numeric BEFORE correlation
covariates = ['Age', 'Gender', 'RIN_novogene', 'PMIhrs', 'ast', 'end', 'mic', 'neu', 'oli', 'opc',' W_1', 'W_2']

# Columns to convert
phenotype_cols = ['Overdose', 'Psych_Diagnosis', 'AUD', 'OUD', 'CUD',
                  'Amphetamine', 'Benzodiazepine', 'Cannabis',
                  'Suicide', 'MDD', 'BD']
cols_to_convert = phenotype_cols + ['Gender']

for col in cols_to_convert:
    if col in binary_df.columns:
        # If bool, convert directly to int
        if binary_df[col].dtype == 'bool':
            binary_df[col] = binary_df[col].astype(int)
        # If object/string, convert 'Yes'/'No' and also 'Male'/'Female' for Gender
        elif binary_df[col].dtype == 'object':
            if col == 'Gender':
                binary_df[col] = binary_df[col].map({'Male': 1, 'Female': 0})
            else:
                binary_df[col] = binary_df[col].map({'Yes': 1, 'No': 0})
        else:
            # If already numeric, no change
            pass

# Quick check of unique values after conversion
for col in cols_to_convert:
    if col in binary_df.columns:
        print(f"{col} unique values: {binary_df[col].unique()}")

## Running models
# Adding a dataframe to store the results
results = []
# Running models
for clock in brain_clocks:
    for pheno in phenotype_cols:
        try:
            # Formula: clock is outcome, phenotype is predictor
            formula = f"{clock} ~ {pheno} + {' + '.join(covariates)}"
            
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                warnings.filterwarnings("ignore", category=RuntimeWarning)

                model = smf.ols(formula=formula, data=binary_df).fit()

            coef = model.params[pheno]
            pval = model.pvalues[pheno]
            conf_int = model.conf_int().loc[pheno]
            tval = model.tvalues[pheno]

            results.append({
                'Clock': clock,
                'Phenotype': pheno,
                'Beta': coef,
                'SE': model.bse[pheno],
                't-value': tval,
                'P-value': pval,
                'CI_lower_95': conf_int[0],
                'CI_upper_95': conf_int[1],
                'R2': model.rsquared,
                'Adj_R2': model.rsquared_adj,
                'N': int(model.nobs)
            })
        
        except Exception as e:
            print(f"Model failed for {clock} ~ {pheno}: {e}")
            results.append({
                'Clock': clock,
                'Phenotype': pheno,
                'Beta': np.nan,
                'SE': np.nan,
                't-value': np.nan,
                'P-value': np.nan,
                'CI_lower_95': np.nan,
                'CI_upper_95': np.nan,
                'R2': np.nan,
                'Adj_R2': np.nan,
                'N': np.nan
            })

results_df = pd.DataFrame(results)

Overdose unique values: [0 1]
Psych_Diagnosis unique values: [0 1]
AUD unique values: [0 1]
OUD unique values: [0 1]
CUD unique values: [0 1]
Amphetamine unique values: [0 1]
Benzodiazepine unique values: [0 1]
Cannabis unique values: [0 1]
Suicide unique values: [0 1]
MDD unique values: [0 1]
BD unique values: [0 1]
Gender unique values: [1 0]


In [7]:
# Save results with tissue information
results_df.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/02uthealth_assoc-09012025.csv", index=False)
results_df

,Clock,Phenotype,Beta,SE,t-value,P-value,CI_lower_95,CI_upper_95,R2,Adj_R2,N
0,DESeq2_brain_clock,Overdose,0.305434,0.156851,1.947291,0.055757,-0.007729,0.618598,0.711090,0.654184,80
1,DESeq2_brain_clock,Psych_Diagnosis,0.217676,0.167826,1.297028,0.199137,-0.117401,0.552752,0.702085,0.643404,80
2,DESeq2_brain_clock,AUD,-0.053135,0.152994,-0.347303,0.729469,-0.358599,0.252328,0.695048,0.634982,80
3,DESeq2_brain_clock,OUD,-0.088607,0.182931,-0.484373,0.629725,-0.453839,0.276626,0.695573,0.635610,80
4,DESeq2_brain_clock,CUD,0.043043,0.175685,0.245002,0.807215,-0.307723,0.393809,0.694769,0.634647,80
...,...,...,...,...,...,...,...,...,...,...,...
160,Predicted_DeepLearning_Stochastic_brain,Benzodiazepine,-0.380180,0.231941,-1.639127,0.105945,-0.843265,0.082904,0.531219,0.438884,80
161,Predicted_DeepLearning_Stochastic_brain,Cannabis,0.124770,0.245189,0.508872,0.612539,-0.364765,0.614305,0.514043,0.418324,80
162,Predicted_DeepLearning_Stochastic_brain,Suicide,-0.381356,0.472317,-0.807415,0.422328,-1.324367,0.561655,0.516908,0.421753,80
163,Predicted_DeepLearning_Stochastic_brain,MDD,0.319448,0.219935,1.452467,0.151110,-0.119666,0.758562,0.527247,0.434130,80


In [8]:
#------------------------------------
# Define clocks and phenotype columns
#------------------------------------
phenotype_cols = ['Overdose', 'Psych_Diagnosis', 'AUD', 'OUD', 'CUD', 'Amphetamine', 'Benzodiazepine', 'Cannabis', 'Suicide', 'MDD', 'BD']
covariates = ['Age', 'Gender', 'RIN_novogene', 'PMIhrs', 'ast', 'end', 'mic', 'neu', 'oli', 'opc',' W_1', 'W_2']
# Strip names in phenotype_cols and covariates
binary_df.columns = binary_df.columns.str.strip()
phenotype_cols = [c.strip() for c in phenotype_cols]
covariates = [c.strip() for c in covariates]

# -------------------------------
# Define a function for stepwise regression for each brain clock
# -------------------------------
def forward_stepwise_selection(data, response, predictors, covariates=[], 
                               threshold_in=0.05, verbose=False):
    """
    Forward stepwise regression: add predictors if p-value < threshold_in.
    Covariates are always included.
    """
    included = list(covariates)  # start with covariates
    # Remove covariates from predictors list to avoid adding again
    predictors = [p for p in predictors if p not in covariates]

    while True:
        changed = False
        excluded = [p for p in predictors if p not in included]
        new_pvals = pd.Series(dtype=float)

        for new_col in excluded:
            try:
                formula = "{} ~ {}".format(response, " + ".join(included + [new_col]))
                model = smf.ols(formula=formula, data=data).fit()
                if new_col in model.pvalues:
                    new_pvals.loc[new_col] = model.pvalues[new_col]
            except Exception:
                continue

        if not new_pvals.empty:
            best_pval = new_pvals.min()
            if best_pval < threshold_in:
                best_feature = new_pvals.idxmin()
                included.append(best_feature)
                changed = True
                if verbose:
                    print(f"Add {best_feature} with p={best_pval:.6f}")

        if not changed:
            break

    return included

In [9]:
# -------------------------------
# Run forward stepwise for each brain clock
# -------------------------------
results_step = []
for clock in brain_clocks:
    try:
        selected = forward_stepwise_selection(
            data=binary_df,
            response=clock,
            predictors=phenotype_cols,
            covariates=covariates,
            verbose=False
        )

        if not selected:  # safety check
            print(f"No predictors selected for {clock}")
            continue

        formula = f"{clock} ~ {' + '.join(selected)}"
        model = smf.ols(formula=formula, data=binary_df).fit()

        for var in selected:
            if var == "Intercept":
                continue
            try:
                coef = model.params[var]
                pval = model.pvalues[var]
                conf_int = model.conf_int().loc[var]
                tval = model.tvalues[var]

                results_step.append({
                    'Clock': clock,
                    'Variable': var,
                    'Beta': coef,
                    'SE': model.bse[var],
                    't-value': tval,
                    'P-value': pval,
                    'CI_lower_95': conf_int[0],
                    'CI_upper_95': conf_int[1],
                    'R2': model.rsquared,
                    'Adj_R2': model.rsquared_adj,
                    'N': int(model.nobs),
                    'Selected_Model': formula
                })
            except KeyError:
                print(f"⚠️ Variable {var} not found in model for {clock}")

    except Exception as e:
        print(f"Stepwise failed for {clock}: {e}")
        results_step.append({
            'Clock': clock,
            'Variable': None,
            'Beta': np.nan,
            'SE': np.nan,
            't-value': np.nan,
            'P-value': np.nan,
            'CI_lower_95': np.nan,
            'CI_upper_95': np.nan,
            'R2': np.nan,
            'Adj_R2': np.nan,
            'N': np.nan,
            'Selected_Model': None
        })

# -------------------------------
# Save results
# -------------------------------
results_df_step = pd.DataFrame(results_step)

In [14]:
# Save results with tissue information
results_df_step.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/02uthealth_assoc_forward-09012025.csv", index=False)
results_df_step

,Clock,Variable,Beta,SE,t-value,P-value,CI_lower_95,CI_upper_95,R2,Adj_R2,N,Selected_Model
0,DESeq2_brain_clock,Age,0.780319,0.075690,10.309459,1.838911e-15,0.629242,0.931397,0.694491,0.639773,80,DESeq2_brain_clock ~ Age + Gender + RIN_novoge...
1,DESeq2_brain_clock,Gender,-0.205816,0.163917,-1.255608,2.136195e-01,-0.532997,0.121365,0.694491,0.639773,80,DESeq2_brain_clock ~ Age + Gender + RIN_novoge...
2,DESeq2_brain_clock,RIN_novogene,0.081435,0.083039,0.980688,3.302749e-01,-0.084311,0.247181,0.694491,0.639773,80,DESeq2_brain_clock ~ Age + Gender + RIN_novoge...
3,DESeq2_brain_clock,PMIhrs,-0.001757,0.070751,-0.024830,9.802644e-01,-0.142977,0.139463,0.694491,0.639773,80,DESeq2_brain_clock ~ Age + Gender + RIN_novoge...
4,DESeq2_brain_clock,ast,0.041277,0.084200,0.490232,6.255715e-01,-0.126786,0.209341,0.694491,0.639773,80,DESeq2_brain_clock ~ Age + Gender + RIN_novoge...
...,...,...,...,...,...,...,...,...,...,...,...,...
184,Predicted_DeepLearning_Stochastic_brain,neu,0.259114,0.125774,2.060162,4.327008e-02,0.008069,0.510160,0.512136,0.424758,80,Predicted_DeepLearning_Stochastic_brain ~ Age ...
185,Predicted_DeepLearning_Stochastic_brain,oli,0.489283,0.545081,0.897634,3.725952e-01,-0.598703,1.577270,0.512136,0.424758,80,Predicted_DeepLearning_Stochastic_brain ~ Age ...
186,Predicted_DeepLearning_Stochastic_brain,opc,0.368898,0.572856,0.643963,5.217982e-01,-0.774528,1.512324,0.512136,0.424758,80,Predicted_DeepLearning_Stochastic_brain ~ Age ...
187,Predicted_DeepLearning_Stochastic_brain,W_1,0.241582,0.087716,2.754135,7.568536e-03,0.066500,0.416664,0.512136,0.424758,80,Predicted_DeepLearning_Stochastic_brain ~ Age ...


In [11]:
#------------------------------------
# Running models with Age squared
#------------------------------------
# Convert Gender (and any other categorical covariates) to numeric BEFORE correlation
covariates = ['Age', 'Age2',  'Gender', 'RIN_novogene', 'PMIhrs', 'ast', 'end', 'mic', 'neu', 'oli', 'opc',' W_1', 'W_2']
# Add the Age2 variable
binary_df['Age2'] = binary_df['Age']**2
# Strip names in phenotype_cols and covariates
binary_df.columns = binary_df.columns.str.strip()
phenotype_cols = [c.strip() for c in phenotype_cols]
covariates = [c.strip() for c in covariates]


# -------------------------------
# Run forward stepwise for each brain clock
# -------------------------------
results_step_age2 = []
for clock in brain_clocks:
    try:
        selected = forward_stepwise_selection(
            data=binary_df,
            response=clock,
            predictors=phenotype_cols,
            covariates=covariates,
            verbose=False
        )

        if not selected:  # safety check
            print(f"No predictors selected for {clock}")
            continue

        formula = f"{clock} ~ {' + '.join(selected)}"
        model = smf.ols(formula=formula, data=binary_df).fit()

        for var in selected:
            if var == "Intercept":
                continue
            try:
                coef = model.params[var]
                pval = model.pvalues[var]
                conf_int = model.conf_int().loc[var]
                tval = model.tvalues[var]

                results_step_age2.append({
                    'Clock': clock,
                    'Variable': var,
                    'Beta': coef,
                    'SE': model.bse[var],
                    't-value': tval,
                    'P-value': pval,
                    'CI_lower_95': conf_int[0],
                    'CI_upper_95': conf_int[1],
                    'R2': model.rsquared,
                    'Adj_R2': model.rsquared_adj,
                    'N': int(model.nobs),
                    'Selected_Model': formula
                })
            except KeyError:
                print(f"⚠️ Variable {var} not found in model for {clock}")

    except Exception as e:
        print(f"Stepwise failed for {clock}: {e}")
        results_step_age2.append({
            'Clock': clock,
            'Variable': None,
            'Beta': np.nan,
            'SE': np.nan,
            't-value': np.nan,
            'P-value': np.nan,
            'CI_lower_95': np.nan,
            'CI_upper_95': np.nan,
            'R2': np.nan,
            'Adj_R2': np.nan,
            'N': np.nan,
            'Selected_Model': None
        })

# -------------------------------
# Save results
# -------------------------------
results_df_step_age2 = pd.DataFrame(results_step_age2)

In [12]:
# Save results with tissue information
results_df_step_age2.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/02uthealth_assoc_forward_age2-09012025.csv", index=False)
results_df_step_age2

,Clock,Variable,Beta,SE,t-value,P-value,CI_lower_95,CI_upper_95,R2,Adj_R2,N,Selected_Model
0,DESeq2_brain_clock,Age,0.737169,0.069729,10.571954,7.718330e-16,0.597951,0.876387,0.751102,0.702077,80,DESeq2_brain_clock ~ Age + Age2 + Gender + RIN...
1,DESeq2_brain_clock,Age2,-0.235078,0.060674,-3.874477,2.482229e-04,-0.356217,-0.113940,0.751102,0.702077,80,DESeq2_brain_clock ~ Age + Age2 + Gender + RIN...
2,DESeq2_brain_clock,Gender,-0.128154,0.150411,-0.852026,3.972806e-01,-0.428460,0.172151,0.751102,0.702077,80,DESeq2_brain_clock ~ Age + Age2 + Gender + RIN...
3,DESeq2_brain_clock,RIN_novogene,0.028368,0.076749,0.369616,7.128521e-01,-0.124867,0.181602,0.751102,0.702077,80,DESeq2_brain_clock ~ Age + Age2 + Gender + RIN...
4,DESeq2_brain_clock,PMIhrs,0.005505,0.064370,0.085520,9.321066e-01,-0.123014,0.134024,0.751102,0.702077,80,DESeq2_brain_clock ~ Age + Age2 + Gender + RIN...
...,...,...,...,...,...,...,...,...,...,...,...,...
202,Predicted_DeepLearning_Stochastic_brain,neu,0.256035,0.126560,2.023035,4.712346e-02,0.003350,0.508719,0.514350,0.418691,80,Predicted_DeepLearning_Stochastic_brain ~ Age ...
203,Predicted_DeepLearning_Stochastic_brain,oli,0.492632,0.547982,0.898993,3.719244e-01,-0.601449,1.586713,0.514350,0.418691,80,Predicted_DeepLearning_Stochastic_brain ~ Age ...
204,Predicted_DeepLearning_Stochastic_brain,opc,0.359685,0.576114,0.624330,5.345630e-01,-0.790563,1.509933,0.514350,0.418691,80,Predicted_DeepLearning_Stochastic_brain ~ Age ...
205,Predicted_DeepLearning_Stochastic_brain,W_1,0.243709,0.088263,2.761180,7.450442e-03,0.067487,0.419931,0.514350,0.418691,80,Predicted_DeepLearning_Stochastic_brain ~ Age ...
